# QIH-PAT Demo — Beaconing / Periodic Anomaly Detection (Synthetic)

This notebook compares a small baseline classifier to one augmented with **QIH-PAT** features:
- Generate a sequence with **intermittent beaconing** (jitter & losses) and non-beacon background.
- Slice into windows; label windows containing beacon behavior.
- Features:
  1. Baseline: simple time-domain stats (mean, std, max, kurtosis-ish).
  2. **QIH-PAT**: analytic QFT histogram for estimated period per window.
- Train a small classifier and report accuracy / ROC-AUC.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quantum_hybrid_system import PeriodicState
from qih_pat import qih_pat_sequence_features

# ---- Data generation ----
rng = np.random.default_rng(0)
T = 100000
timeline = rng.normal(0, 0.2, size=T)

def inject_beacons(arr, interval=120, jitter=10, amp=2.0, p_loss=0.1):
    t = 0
    while t < len(arr):
        if rng.random() > p_loss:
            arr[t] += amp + 0.2*rng.normal()
        t += max(1, int(interval + rng.integers(-jitter, jitter+1)))

# Half the sequence has beaconing
timeline2 = timeline.copy()
inject_beacons(timeline2, interval=120, jitter=15, amp=2.5, p_loss=0.15)

# Windowing & labels
W, S = 256, 128
def windows(x):
    for start in range(0, len(x)-W+1, S):
        yield x[start:start+W]

X0 = np.vstack(list(windows(timeline)))
X1 = np.vstack(list(windows(timeline2)))
y0 = np.zeros(len(X0), dtype=int)
y1 = np.ones(len(X1), dtype=int)

X = np.vstack([X0, X1])
y = np.concatenate([y0, y1])

perm = rng.permutation(len(X))
X, y = X[perm], y[perm]

# ---- Baseline features (cheap time-domain) ----
def baseline_feats(Wx):
    # Wx: [N, W]
    m = Wx.mean(axis=1)
    s = Wx.std(axis=1)
    mx = Wx.max(axis=1)
    mn = Wx.min(axis=1)
    p95 = np.percentile(Wx, 95, axis=1)
    return np.vstack([m, s, mx, mn, p95]).T

Xb = baseline_feats(X)

# ---- QIH-PAT features per window ----
H, periods = qih_pat_sequence_features(X, win=W, stride=W, n=10, shots=1024, bins=64)

# ---- Train/test split ----
N = len(X)
split = int(0.8 * N)
Xb_tr, Xb_te = Xb[:split], Xb[split:]
H_tr, H_te = H[:split], H[split:]
y_tr, y_te = y[:split], y[split:]

# ---- Classifiers ----
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score
    base = LogisticRegression(max_iter=200).fit(Xb_tr, y_tr)
    qih  = LogisticRegression(max_iter=200).fit(np.hstack([Xb_tr, H_tr]), y_tr)
    pb = base.predict(Xb_te)
    pq = qih.predict(np.hstack([Xb_te, H_te]))
    pbp = base.predict_proba(Xb_te)[:,1]
    pqp = qih.predict_proba(np.hstack([Xb_te, H_te]))[:,1]
    acc_base = accuracy_score(y_te, pb)
    acc_qih = accuracy_score(y_te, pq)
    auc_base = roc_auc_score(y_te, pbp)
    auc_qih = roc_auc_score(y_te, pqp)
    print({"acc_base": float(acc_base), "acc_qih": float(acc_qih), "auc_base": float(auc_base), "auc_qih": float(auc_qih)})
except Exception as e:
    print("sklearn not available:", e)

# ---- Visualization ----
import matplotlib.pyplot as plt
plt.figure()
plt.plot(H_tr[y_tr==0].mean(axis=0), label="non-beacon")
plt.plot(H_tr[y_tr==1].mean(axis=0), label="beacon")
plt.title("Mean QIH-PAT histogram per class (train)")
plt.xlabel("bin")
plt.ylabel("probability")
plt.legend()
plt.show()